# WalkingDict — RAG Corpus EDA

Exploratory analysis of the unified dictionary corpus that feeds the RAG index.

Extra deps beyond `requirements.txt`:
```
pip install matplotlib seaborn upsetplot scipy
# optional, for the embedding-space cell:
pip install umap-learn scikit-learn
```

In [ ]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(context="notebook", style="whitegrid")
pd.set_option("display.max_colwidth", 120)

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED = REPO / "data" / "processed"
CORPUS = PROCESSED / "unified_corpus.jsonl"
FIGS = REPO / "notebooks" / "figs"
FIGS.mkdir(exist_ok=True, parents=True)
print("corpus:", CORPUS, "exists:", CORPUS.exists())

## 1. Load corpus

`df_doc` = one row per `(word, source)` entry. `df` = one row per sense (after exploding).

In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]

docs = load_jsonl(CORPUS)
df_doc = pd.DataFrame(docs)
df_doc["senses_count"] = df_doc["senses"].apply(len)
df_doc["word_len"] = df_doc["word"].str.len()
df_doc["has_ipa"] = df_doc["ipa"].apply(lambda v: bool(v) if isinstance(v, list) else False)
df_doc["has_etymology"] = df_doc["etymology"].fillna("").str.strip().astype(bool)
df_doc["n_synonyms"] = df_doc.get("synonyms", pd.Series([[]]*len(df_doc))).apply(lambda v: len(v or []))
df_doc["n_antonyms"] = df_doc.get("antonyms", pd.Series([[]]*len(df_doc))).apply(lambda v: len(v or []))
print(df_doc.shape, "docs")
df_doc.head(3)

In [ ]:
# Sense-level dataframe (one row per (word, source, sense))
df = df_doc.explode("senses", ignore_index=True)
df = pd.concat([df.drop(columns=["senses"]), df["senses"].apply(pd.Series)], axis=1)
df["def_len_chars"] = df["definition"].fillna("").str.len()
df["def_len_tokens"] = df["definition"].fillna("").str.split().apply(len)
df["n_examples"] = df["examples"].apply(lambda v: len(v) if isinstance(v, list) else 0)
df["has_example"] = df["n_examples"] > 0
print(df.shape, "senses")
df[["word", "source", "part_of_speech", "def_len_tokens", "n_examples"]].head()

## §5.1 · Scale and source composition

In [ ]:
scale = pd.DataFrame({
    "docs": df_doc["source"].value_counts(),
    "senses": df["source"].value_counts(),
    "unique_words": df_doc.groupby("source")["word"].nunique(),
})
scale.loc["TOTAL"] = scale.sum()
scale

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
df_doc["source"].value_counts().plot.bar(ax=ax[0], color="steelblue")
ax[0].set(title="Entries per source", ylabel="docs")
df["source"].value_counts().plot.bar(ax=ax[1], color="indianred")
ax[1].set(title="Senses per source", ylabel="senses")
plt.tight_layout(); plt.savefig(FIGS / "source_counts.png", dpi=150); plt.show()

## §5.2 · Distributions

### Part-of-speech per source

In [ ]:
pos_tab = pd.crosstab(df["source"], df["part_of_speech"]).fillna(0)
pos_tab_norm = pos_tab.div(pos_tab.sum(axis=1), axis=0)
pos_tab_norm.plot.bar(stacked=True, figsize=(10, 4), colormap="tab20")
plt.ylabel("share"); plt.title("POS mix by source")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout(); plt.savefig(FIGS / "pos_by_source.png", dpi=150); plt.show()
pos_tab

### Senses per word (polysemy)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for src, sub in df_doc.groupby("source"):
    ax.hist(sub["senses_count"], bins=range(1, 30), alpha=0.5, label=src)
ax.set(xlabel="senses per entry", ylabel="# entries", yscale="log",
       title="Polysemy distribution (log y)")
ax.legend(); plt.tight_layout(); plt.savefig(FIGS / "polysemy.png", dpi=150); plt.show()
df_doc.groupby("source")["senses_count"].describe().round(2)

### Definition length & example availability

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df[df["def_len_tokens"] < df["def_len_tokens"].quantile(0.99)],
            x="source", y="def_len_tokens", ax=ax[0])
ax[0].set(title="Definition length (tokens, 99th-pct clipped)")
(df.groupby("source")["has_example"].mean()
   .sort_values(ascending=False)
   .plot.bar(ax=ax[1], color="seagreen"))
ax[1].set(title="Fraction of senses with ≥1 example", ylim=(0, 1))
plt.tight_layout(); plt.savefig(FIGS / "def_and_examples.png", dpi=150); plt.show()

### Metadata coverage (IPA / etymology / synonyms)

In [ ]:
cov = df_doc.groupby("source")[["has_ipa", "has_etymology"]].mean()
cov["has_synonym"] = df_doc.groupby("source")["n_synonyms"].apply(lambda s: (s > 0).mean())
cov["has_antonym"] = df_doc.groupby("source")["n_antonyms"].apply(lambda s: (s > 0).mean())
cov.plot.bar(figsize=(9, 4)); plt.ylim(0, 1); plt.ylabel("coverage")
plt.title("Metadata coverage by source"); plt.legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIGS / "coverage.png", dpi=150); plt.show()
cov.round(3)

### Word-length distribution

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(data=df_doc, x="word_len", hue="source", bins=40, element="step", stat="density")
plt.title("Word-length distribution by source")
plt.tight_layout(); plt.savefig(FIGS / "word_len.png", dpi=150); plt.show()

### Vocabulary overlap across sources

In [ ]:
from itertools import combinations

vocab = {s: set(g["word"].str.lower()) for s, g in df_doc.groupby("source")}
print({s: len(v) for s, v in vocab.items()})
for a, b in combinations(vocab, 2):
    inter = len(vocab[a] & vocab[b])
    union = len(vocab[a] | vocab[b])
    print(f"{a} ∩ {b}: {inter:>5}  |  Jaccard: {inter/union:.3f}")

# Optional UpSet plot (requires: pip install upsetplot)
try:
    from upsetplot import from_contents, UpSet
    data = from_contents(vocab)
    UpSet(data, show_counts=True).plot()
    plt.suptitle("Vocabulary overlap across sources")
    plt.savefig(FIGS / "vocab_overlap.png", dpi=150, bbox_inches="tight")
    plt.show()
except ImportError:
    print("upsetplot not installed — skipping plot")

### Label distributions (`category`, `difficulty`)

These are the two categorical labels assigned during unification — treat as targets for correlation analysis.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
df_doc["category"].value_counts().plot.bar(ax=ax[0], color="slateblue")
ax[0].set(title="Category")
df_doc["difficulty"].value_counts().plot.bar(ax=ax[1], color="darkorange")
ax[1].set(title="Difficulty")
plt.tight_layout(); plt.savefig(FIGS / "labels.png", dpi=150); plt.show()

pd.crosstab(df_doc["source"], df_doc["difficulty"], normalize="index").round(3)

## §5.2 · Feature correlations

### Numeric: Spearman heatmap

In [ ]:
num = df_doc[["senses_count", "word_len", "n_synonyms", "n_antonyms"]].copy()
num["avg_def_tokens"] = df.groupby(["word", "source"])["def_len_tokens"].mean().reset_index(drop=True)
num["avg_examples"]  = df.groupby(["word", "source"])["n_examples"].mean().reset_index(drop=True)
corr = num.corr(method="spearman")
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Spearman correlation (entry-level features)")
plt.tight_layout(); plt.savefig(FIGS / "corr_numeric.png", dpi=150); plt.show()

### Categorical vs label: Cramér's V

How strongly does each categorical feature predict `difficulty` / `category`?

In [ ]:
from scipy.stats import chi2_contingency

def cramers_v(a: pd.Series, b: pd.Series) -> float:
    tab = pd.crosstab(a, b)
    chi2 = chi2_contingency(tab, correction=False)[0]
    n = tab.values.sum()
    r, k = tab.shape
    denom = n * (min(r, k) - 1)
    return float(np.sqrt(chi2 / denom)) if denom else np.nan

# Use sense-level df so part_of_speech is in scope
cat_features = ["source", "part_of_speech", "has_example"]
targets = ["difficulty", "category"]
rows = []
for feat in cat_features:
    for tgt in targets:
        rows.append({"feature": feat, "target": tgt,
                     "cramers_v": cramers_v(df[feat].astype(str), df[tgt].astype(str))})
pd.DataFrame(rows).pivot(index="feature", columns="target", values="cramers_v").round(3)

### Numeric features vs `difficulty` (ordinal)

Encode difficulty as `beginner=0, intermediate=1, advanced=2` and compute Spearman.

In [ ]:
diff_order = {"beginner": 0, "intermediate": 1, "advanced": 2}
df_doc["difficulty_ord"] = df_doc["difficulty"].map(diff_order)
numeric_cols = ["senses_count", "word_len", "n_synonyms", "n_antonyms"]
(df_doc[numeric_cols + ["difficulty_ord"]]
   .corr(method="spearman")["difficulty_ord"]
   .drop("difficulty_ord").sort_values(key=abs, ascending=False).round(3))

## (Optional) Embedding-space exploration

Sample vectors straight from ChromaDB, reduce with UMAP, color by source. Lets you argue *"each source covers a distinct semantic neighborhood — multi-source retrieval is non-redundant."*

In [ ]:
# Requires: pip install chromadb umap-learn scikit-learn
try:
    import chromadb
    import umap

    client = chromadb.PersistentClient(path=str(REPO / "chroma_db"))
    cols = client.list_collections()
    print("collections:", [c.name for c in cols])
    col = cols[0]
    got = col.get(include=["embeddings", "metadatas"], limit=3000)
    emb = np.array(got["embeddings"])
    meta = pd.DataFrame(got["metadatas"])
    print("sampled:", emb.shape)

    red = umap.UMAP(n_neighbors=20, min_dist=0.1, random_state=0).fit_transform(emb)
    plt.figure(figsize=(8, 6))
    for src, grp in meta.assign(x=red[:, 0], y=red[:, 1]).groupby("source"):
        plt.scatter(grp["x"], grp["y"], s=5, alpha=0.5, label=src)
    plt.legend(markerscale=3); plt.title("UMAP of corpus embeddings, colored by source")
    plt.tight_layout(); plt.savefig(FIGS / "umap_source.png", dpi=150); plt.show()
except Exception as e:
    print("skipped:", e)

## Scratch: summary table for the report

Drop this block into §5.1 as-is.

In [ ]:
summary = pd.DataFrame({
    "docs": df_doc["source"].value_counts(),
    "senses": df["source"].value_counts(),
    "unique_words": df_doc.groupby("source")["word"].nunique(),
    "avg_senses/doc": df_doc.groupby("source")["senses_count"].mean().round(2),
    "avg_def_tokens": df.groupby("source")["def_len_tokens"].mean().round(1),
    "example_rate": df.groupby("source")["has_example"].mean().round(3),
    "ipa_rate": df_doc.groupby("source")["has_ipa"].mean().round(3),
    "etym_rate": df_doc.groupby("source")["has_etymology"].mean().round(3),
})
print(summary.to_markdown())
summary